# Video Analysis: Action Recognition with SlowFast and ViViT

This session is about classifying actions in video – which turns out to be surprisingly useful for CSS once you think about the kind of material we typically work with: speeches, debates, protest footage, campaign ads. We'll look at two pretrained models that do this: **SlowFast** and **ViViT**.

## What is SlowFast?

SlowFast (Feichtenhofer et al., 2019) processes video through two parallel pathways:
- **Slow Pathway**: processes fewer frames at high spatial resolution – good at recognizing objects and scenes
- **Fast Pathway**: processes many frames at lower spatial resolution – good at capturing motion dynamics

Both pathways see the same video clip but at different temporal resolutions, and they exchange information via lateral connections (sideways connections between the two parallel streams – the fast pathway passes motion information across into the slow pathway at multiple intermediate layers, not just at the very end). The combination makes SlowFast work well for action recognition tasks (e.g. "speaking", "waving", "applauding").

Last week we briefly discussed image pyramids – the idea of representing the same image at multiple spatial scales simultaneously, because coarse and fine scales carry different information. SlowFast applies exactly the same intuition, but along the time dimension instead of space: two temporal resolutions of the same video, processed in parallel. The slow pathway is the "coarse" view (few frames, high spatial detail), the fast pathway is the "fine" temporal view (many frames, lower spatial detail). So SlowFast is essentially a temporal pyramid with lateral connections between the two streams.

## Pretrained on Kinetics-400

Both models here are pretrained on Kinetics-400 – 400 everyday action categories like cooking, dancing, or sports. There are no political categories in there. Everything the model outputs is one of these 400 classes, which is worth keeping in mind when interpreting results on CSS material.

Reference: https://pytorchvideo.org/docs/tutorial_torchhub_inference

In [1]:
!sed -i 's/import torchvision.transforms.functional_tensor as F_t/import torchvision.transforms.functional as F_t/' \
  ~/miniconda3/lib/python3.13/site-packages/pytorchvideo/transforms/augmentations.py


sed: 1: "/Users/clarafochler/min ...": command c expects \ followed by text


In [2]:
import torch
import json
from torchvision.transforms import Compose, Lambda
from torchvision.transforms._transforms_video import (
    CenterCropVideo,
    NormalizeVideo,
)
from pytorchvideo.data.encoded_video import EncodedVideo
from pytorchvideo.transforms import (
    ApplyTransformToKey,
    ShortSideScale,
    UniformTemporalSubsample,
)

/Users/clarafochler/miniconda3/lib/python3.13/site-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/Users/clarafochler/miniconda3/lib/python3.13/site-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(
objc[67174]: Class AVFFrameReceiver is implemented in both /Users/clarafochler/miniconda3/lib/python3.13/site-packages/av/.dylibs/libavdevice.62.3.101.dylib (0x126acc3a8) and /Users/clarafochler/miniconda3/lib/python3.13/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x16a9843a8). This may cause spurious casting failures and mysterious crashes. One of the duplicate

In [3]:
# Use GPU if available - makes a big difference for longer videos
device = "cuda" if torch.cuda.is_available() else "cpu"

# SlowFast R50: ResNet-50-based SlowFast model, pretrained on Kinetics-400
slowfast_model_name = "slowfast_r50"
slowfast_model = torch.hub.load("facebookresearch/pytorchvideo", model=slowfast_model_name, pretrained=True)

# important: always set model to eval mode for inference
slowfast_model = slowfast_model.to(device)
slowfast_model = slowfast_model.eval()

Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /Users/clarafochler/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/SLOWFAST_8x8_R50.pyth" to /Users/clarafochler/.cache/torch/hub/checkpoints/SLOWFAST_8x8_R50.pyth


100%|██████████| 264M/264M [00:25<00:00, 10.8MB/s] 


In [21]:
slowfast_model 

Net(
  (blocks): ModuleList(
    (0): MultiPathWayWithFuse(
      (multipathway_blocks): ModuleList(
        (0): ResNetBasicStem(
          (conv): Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False)
          (norm): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation): ReLU()
          (pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=[0, 1, 1], dilation=1, ceil_mode=False)
        )
        (1): ResNetBasicStem(
          (conv): Conv3d(3, 8, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False)
          (norm): BatchNorm3d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation): ReLU()
          (pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=[0, 1, 1], dilation=1, ceil_mode=False)
        )
      )
      (multipathway_fusion): FuseFastToSlow(
        (conv_fast_to_slow): Conv3d(8, 16, kernel_size=(7, 1, 1), st

In [6]:
!curl -L -o kinetics_classnames.json "https://dl.fbaipublicfiles.com/pyslowfast/dataset/class_names/kinetics_classnames.json"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10326  100 10326    0     0  17191      0 --:--:-- --:--:-- --:--:-- 17210


In [22]:
with open("kinetics_classnames.json", "r") as f:
    kinetics_classnames = json.load(f)

# Build a mapping from numeric index to human-readable class name
kinetics_id_to_classname = {}
for k, v in kinetics_classnames.items():
    kinetics_id_to_classname[v] = str(k).replace('"', "")


In [23]:
kinetics_id_to_classname

{290: 'sharpening knives',
 115: 'eating ice cream',
 81: 'cutting nails',
 53: 'changing wheel',
 19: 'bench pressing',
 88: 'deadlifting',
 111: 'eating carrots',
 192: 'marching',
 358: 'throwing discus',
 231: 'playing flute',
 72: 'cooking on campfire',
 33: 'breading or breadcrumbing',
 218: 'playing badminton',
 276: 'ripping paper',
 244: 'playing saxophone',
 197: 'milking cow',
 169: 'juggling balls',
 130: 'flying kite',
 43: 'capoeira',
 187: 'making jewelry',
 100: 'drinking',
 228: 'playing cymbals',
 61: 'cleaning gutters',
 161: 'hurling (sport)',
 239: 'playing organ',
 361: 'tossing coin',
 395: 'wrestling',
 103: 'driving car',
 150: 'headbutting',
 147: 'gymnastics tumbling',
 186: 'making bed',
 0: 'abseiling',
 155: 'holding snake',
 278: 'rock climbing',
 71: 'cooking egg',
 182: 'long jump',
 17: 'bee keeping',
 365: 'trimming or shaving beard',
 63: 'cleaning shoes',
 86: 'dancing gangnam style',
 50: 'catching or throwing softball',
 164: 'ice skating',
 168: 

In [25]:
# SlowFast transform

side_size = 256         
crop_size = 256        
mean = [0.45, 0.45, 0.45]    # per-channel pixel mean (derived from Kinetics training data)
std = [0.225, 0.225, 0.225] 
num_frames = 32         
sampling_rate = 2        # take every 2nd frame from the original video
frames_per_second = 30  
alpha = 4                # slow pathway gets 1/alpha as many frames as the fast pathway (8 vs. 32)

class PackPathway(torch.nn.Module):
    """
    Splits the video clip into fast and slow pathways.
    Fast pathway: all num_frames frames (high temporal resolution)
    Slow pathway: uniformly subsampled to num_frames // alpha frames (high spatial resolution)
    """
    def __init__(self):
        super().__init__()

    def forward(self, frames: torch.Tensor):
        fast_pathway = frames
        # select every alpha-th frame for the slow pathway
        slow_pathway = torch.index_select(
            frames,
            1,
            torch.linspace(
                0, frames.shape[1] - 1, frames.shape[1] // alpha
            ).long(),
        )
        return [slow_pathway, fast_pathway]

transform = ApplyTransformToKey(
    key="video",
    transform=Compose(
        [
            UniformTemporalSubsample(num_frames),  # evenly sample num_frames frames across the clip
            Lambda(lambda x: x/255.0),             # normalize pixel values to [0, 1]
            NormalizeVideo(mean, std),             # z-standardize using Kinetics statistics
            ShortSideScale(size=side_size),        # scale short side to side_size
            CenterCropVideo(crop_size),            # square center crop
            PackPathway()                          # split into slow and fast pathway
        ]
    ),
)

# passed to get_clip() below – tells the loader how many seconds to extract from the video
clip_duration = (num_frames * sampling_rate) / frames_per_second


In [12]:
video_path = "/Users/clarafochler/Documents/Dissertation/Data/robin_wagener_part3_47insteadof48_gruene/7358388148751142177.mp4"

In [13]:
# Start and end point of the clip to analyze
# clip_duration is determined by num_frames and sampling_rate (about 2 seconds with the settings above)
start_sec = 0
end_sec = start_sec + clip_duration

# EncodedVideo reads the video and allows access to specific time segments
video = EncodedVideo.from_path(video_path)
video_data = video.get_clip(start_sec=start_sec, end_sec=end_sec)

# Apply transform: temporal sampling, normalization, pathway split
video_data = transform(video_data)

# [None, ...] adds a batch dimension – models always expect input of shape (batch, ...)
inputs = video_data["video"]
inputs = [i.to(device)[None, ...] for i in inputs]

In [14]:
preds = slowfast_model(inputs)

In [26]:
# Softmax to convert raw model outputs into probabilities
post_act = torch.nn.Softmax(dim=1)
preds = post_act(preds)
pred_classes = preds.topk(k=5).indices

pred_class_names = [kinetics_id_to_classname[int(i)] for i in pred_classes[0]]
print("SlowFast – Top-5 predictions: %s" % ", ".join(pred_class_names))


SlowFast – Top-5 predictions: giving or receiving award, answering questions, auctioning, celebrating, news anchoring


What may be shortcomings of SlowFast?

The Top-5 labels show which Kinetics classes the model considers most likely for the clip. Results can be a bit surprising – "talking" or "giving a speech" do exist in Kinetics-400 and will sometimes show up, but so will completely unrelated categories if there's a lot of background movement or camera shake.

One clip is rarely enough to draw conclusions. In practice you'd run this over many clips and look at distributions, not individual predictions.

# ViViT – Vision Transformer for Video

ViViT (Video Vision Transformer, Arnab et al., 2021) brings the Transformer architecture to video.

We use `google/vivit-b-16x2-kinetics400` here – also trained on Kinetics-400, so the output categories are the same as with SlowFast.

In [16]:
import torch
from transformers import VivitImageProcessor, VivitForVideoClassification

vivit_model_name = "google/vivit-b-16x2-kinetics400"

# The processor handles resizing and normalization – equivalent to the manual transform pipeline above
processor = VivitImageProcessor.from_pretrained(vivit_model_name)

vivit_model = VivitForVideoClassification.from_pretrained(
    vivit_model_name,
    attn_implementation="sdpa",  # efficient attention implementation
    device_map="auto"            # automatically use GPU if available
)

vivit_model.eval()

preprocessor_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/356M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

VivitForVideoClassification(
  (vivit): VivitModel(
    (embeddings): VivitEmbeddings(
      (patch_embeddings): VivitTubeletEmbeddings(
        (projection): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x VivitLayer(
        (attention): VivitAttention(
          (q_proj): Linear(in_features=768, out_features=768, bias=True)
          (k_proj): Linear(in_features=768, out_features=768, bias=True)
          (v_proj): Linear(in_features=768, out_features=768, bias=True)
          (o_proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (layernorm_before): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (layernorm_after): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): VivitMLP(
          (activation_fn): FastGELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): 

model.safetensors:   0%|          | 0.00/356M [00:00<?, ?B/s]

In [17]:
import cv2
import numpy as np

def load_video(path, num_frames=32):
    cap = cv2.VideoCapture(path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Sample uniformly across the entire video, not just the first frames.
    # Without this, a 2-minute video would only be analyzed for its first second.
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)  # jump directly to the target frame
        ret, frame = cap.read()
        if not ret:
            frames.append(np.zeros((224, 224, 3), dtype=np.uint8))
            continue
        frame = cv2.resize(frame, (224, 224))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # OpenCV reads BGR, ViViT expects RGB
        frames.append(frame)

    cap.release()
    return np.array(frames)

frames = load_video(video_path)

In [18]:
inputs = processor(list(frames), return_tensors="pt")
inputs = {k: v.to(vivit_model.device) for k, v in inputs.items()}

with torch.no_grad():  # no gradient needed for inference – saves memory
    outputs = vivit_model(**inputs)

logits = outputs.logits
probs = torch.nn.functional.softmax(logits, dim=1)

In [27]:
topk = torch.topk(probs, k=5)

print("ViViT – Top-5 predictions:")
for i in topk.indices[0]:
    label = vivit_model.config.id2label[i.item()]
    prob = probs[0][i].item()
    print(f"  {label}: {prob:.3f}")

ViViT – Top-5 predictions:
  LABEL_51: 0.273
  LABEL_202: 0.163
  LABEL_254: 0.085
  LABEL_2: 0.064
  LABEL_289: 0.034


In [28]:
topk = torch.topk(probs, k=5)

print("ViViT – Top-5 predictions:")
for i in topk.indices[0]:
    label = kinetics_id_to_classname[i.item()]  # same mapping as SlowFast
    prob = probs[0][i].item()
    print(f"  {label}: {prob:.3f}")


ViViT – Top-5 predictions:
  celebrating: 0.273
  news anchoring: 0.163
  presenting weather forecast: 0.085
  answering questions: 0.064
  shaking head: 0.034


## SlowFast vs. ViViT

Both models classify a clip into one of 400 Kinetics categories - even though they do it very differently.

For CSS use cases neither is obviously better out of the box, since both were trained on Kinetics. The more interesting question is which is easier to adapt. Since ViViT is usable through HuggingFace it makes fine-tuning on custom labels much more straightforward.


Do the Top-5 predictions from SlowFast and ViViT agree? Where do they differ – and what might that tell us about what each model is picking up on?